In [ ]:
import sys
import os.path # REMOVE
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.ticker as ticker

mm = 1.0/2.54/10
repo_root_path = join('..','..')
python_modules_paths = (
    join(repo_root_path, 'src', 'python'),
    'python'
)
for python_modules_path in python_modules_paths:
    if python_modules_path not in sys.path:
        sys.path.append(python_modules_path)

from pilot300W.pilot300W import Pilot300W, MW_fluid, Vn_to_M

from specie_properties.common import AW, MW
from foam.common import read_multiple_df
from foam.foamcase import FoamCase
from specie_properties import janaf
from scm.scm import IlmenitePhase, ilmenite
from specie_properties.janaf_ilmenite import janafs_ilmenite, janafs_ilmenite_orig
plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))

MWs = dict()

# Tests
assert((Vn_to_M('air', 7.2) - 0.000144398)/0.000144398 < 1e-4) # AR
assert((Vn_to_M('Ar',  1.4) - 3.87494e-05)/3.87494e-05 < 1e-4) # FR
assert((Vn_to_M('CO2', 0.3) - 9.14754e-06)/9.14754e-06 < 1e-4) # DC1
assert((Vn_to_M('CO2', 0.1) - 3.04918e-06)/3.04918e-06 < 1e-4) # SL

# Reacting

In [ ]:
p = Pilot300W('pilot300Wv2_270g_880C_Twall_kN4', offset=0.005)

In [ ]:
# Global average temperature (estimate) to track the simulations progress
fig, ax = plt.subplots()

p.domain.register_custom_function_object('T', 'volAverage', )
(p.domain.T() - 273.15).plot(ax=ax)
ax.set(
    xlabel=r'$t$, s',
    ylabel=r'$T_\mathrm{avg}$, °C',
    xlim=(11,None),
    ylim=(894,896),
)
ax.get_legend().remove()
ax.axvline(42)
ax.grid()

In [ ]:
# examples of how post-processing data can be accessed
# p.faceZones['controlSurface_DC_AR'].mdot_sp()
# p.patches['outlet_FR'].mdot_sp()
# p.probes['pressureProbesFn'].get_data()
# p.domain.Tavgest(phase='solids')
# p.cellSets['controlVolume_DC'].m_solid_sp()
# p.cellSets['controlVolume_DC'].Tavgest(phase='solids')
# p.domain.NRR_O2()

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

ax_cellSet_Tavgest = ax #s[0]
offset = 1
window = 100
# t_start = 250 # s
t_start = 1 # s

for location in ['AR', 'FR', 'DC', 'SL']:
    cellSet_name = 'controlVolume_' + location
    print(f'Processing {cellSet_name}')
    (p.cellSets[cellSet_name].Tavgest(phase='solids').rolling(window).mean()-273.15).loc[t_start:].plot(
        ax = ax_cellSet_Tavgest, color='k', label=cellSet_name.split('_')[-1], y='volAverage(rho.solids,alpha.solids,T.solids)')
ax_cellSet_Tavgest.set(
    xlabel=r'$t$, s',
    ylabel=r'$T_\mathrm{avg}$, °C', # note that this is an estimate of the average temperature
    xlim=(12,30),
    ylim=(879, 900)
)

ax.axhline(880, color='k', linestyle='--', label=r'$T_\mathrm{wall}$')

ax_cellSet_Tavgest.text(17.0,898.0,'AR')
ax_cellSet_Tavgest.text(17.5,894.5,'DC')
ax_cellSet_Tavgest.text(18.0,891.5,'FR')
ax_cellSet_Tavgest.text(19.5,889.5,'SL')
ax_cellSet_Tavgest.text(17,880.7,r'$T_\mathrm{wall}$')
ax_cellSet_Tavgest.get_legend().remove()
# ax.grid()
fig.tight_layout()
fig.savefig('plots/pilot300W_temperature.pdf')

## Circulation

## Solid circulation rate

<span style="color:red">Why mismatch? Variable time step! Direct averaging is not possible </span>  

There are fluctuations of opposite fluxes through control surfaces exist, expecially in DC->FR
Apply averaging with care! 

NOTE:
I used boxToFace, but without setAndNormalToFaceZone!
In this case, oriented sum (according to copilot) uses owner cell's location to determine face orientation.  
In constant/polyMesh/faceZones all flip maps are uniform

<span style="color:green">orientFaceZone utility!</span>

According to tests with printFaceZoneOwners utility, all faces should have a correct orientation

In [ ]:
def circulation_rate(c, csname):
    """Get a circulation rate for a given control surface."""
    sign = -1 if csname in ['SL_FR', 'DC_AR'] else 1
    return sign*c.faceZones['controlSurface_' + csname].mdot()['orientedSum(alphaRhoPhi.solids)']

def circulation_rate_avg(c, csname, t_skip_fraction=0.2):
    """Get a circulation rate for all control surfaces."""
    # for csname in ['SL_AR', 'SL_FR', 'DC_AR', 'DC_FR']:
    mdot = circulation_rate(c, csname)
    mdot = mdot.iloc[int(len(mdot)*t_skip_fraction):]
    dt = mdot.index[-1]-mdot.index[0]
    return np.trapz(mdot, x=mdot.index, axis=0)/dt

def circulation_rate_overall(c, t_skip_fraction=0.2):
    """Get a circulation rate for all control surfaces."""
    return np.mean([circulation_rate_avg(c, cs, t_skip_fraction) for cs in ['SL_AR', 'SL_FR', 'DC_AR', 'DC_FR']])

print('Circulation rate, at each surface:')
t_start = 12
t_skip_fraction=0.6

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

for csname in ['SL_AR', 'SL_FR', 'DC_AR', 'DC_FR']:
    mdot = circulation_rate(p, csname).loc[t_start:].rolling(100).mean()
    h = ax.plot(mdot.index, mdot, label=csname)
    mdot_avg = circulation_rate_avg(p, csname, t_skip_fraction=t_skip_fraction)
    ax.plot(
        [mdot.index[0], mdot.index[-1]],
        [mdot_avg, mdot_avg],
        '--', color=h[0].get_color()
    )
    print(f"{csname}: {np.abs(1000*mdot_avg):5.4f} g/s")
    
print(f"Overall circulation rate: {np.abs(1000*circulation_rate_overall(p, t_skip_fraction=t_skip_fraction)):5.4f} g/s")
ax.legend()
ax.set(
    xlim = (t_skip_fraction*30, None),
    ylim=(0.005,0.015)
)

In [ ]:
Ro = 0.033
ndot_H2 = p.mdot_sp('inlet_FR', 'H2').iloc[-1]/0.002 # (kg/s) / (kg/mol) = mol / s
ndot_CO = p.mdot_sp('inlet_FR', 'CO').iloc[-1]/0.028 # (kg/s) / (kg/mol) = mol / s
# 2*H2 + O2 = 2 H2O
# 2*CO + O2 = 2 CO2
ndot_O2_required = ndot_H2 / 2 + ndot_CO / 2 # mol/s
mdot_O2_required = ndot_O2_required * 0.032 # kg/s
print(f'O2 required:       {1000*mdot_O2_required:.5f} g/s')
print(f'ilmenite required: {1000*mdot_O2_required/Ro:.4f} g/s')
print(f'ilmenite with 5% variation required: {1000*mdot_O2_required/Ro*20:.4f} g/s')


In [ ]:
p.faceZones[f'controlSurface_SL_AR'].mdot_solids_sp()[f'orientedSum(surfaceInterpolate(FeO.solids))']

In [ ]:
# using mass flow rates at control surfaces
# WARNING: there was a mistake in function object.
# DO NOT USE THIS
def mdot_sp(sp, cs):
    return p.faceZones[f'controlSurface_{cs}'].mdot_solids_sp()[f'orientedSum(surfaceInterpolate({sp}.solids))']
rho_ratio_ = MW('Fe2O3')/(2*MW('FeO'))

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))
window = 2
# (mdot_sp('Fe2O3', 'SL_AR')/(mdot_sp('Fe2O3', 'SL_AR')+rho_ratio_*mdot_sp('FeO', 'SL_AR'))).rolling(window).mean().plot(ax=ax)
(mdot_sp('Fe2O3', 'SL_FR')/(mdot_sp('Fe2O3', 'SL_FR')+rho_ratio_*mdot_sp('FeO', 'SL_FR'))).rolling(window).mean().plot(ax=ax, color='k')
(mdot_sp('Fe2O3', 'DC_AR')/(mdot_sp('Fe2O3', 'DC_AR')+rho_ratio_*mdot_sp('FeO', 'DC_AR'))).rolling(window).mean().plot(ax=ax, color='k')

# ax.text(290,0.9925,r'AR$\rightarrow$FR', fontsize=11)
# ax.text(300,0.947,r'FR$\rightarrow$AR', fontsize=11)
# (mdot_sp('Fe2O3', 'DC_FR')/(mdot_sp('Fe2O3', 'DC_FR')+rho_ratio_*mdot_sp('FeO', 'DC_FR'))).rolling(window).mean().plot(ax=ax)
ax.set(
    xlabel=r'$t$, s',
    ylabel = 'X, -',
    # xlim = (280, 336),
    ylim = (None, 1)
)
fig.tight_layout()
# fig.savefig('plots/pilot300W_oxidation.pdf')

In [ ]:
# using solids mass in reactors
def m_sp(sp, cv):
    return p.cellSets[f'controlVolume_{cv}'].m_solid_sp()[f'volIntegrate(alpha.solids,rho.solids,{sp}.solids)']

rho_ratio_ = MW('Fe2O3')/(2*MW('FeO'))
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))
(m_sp('Fe2O3', 'AR')/(m_sp('Fe2O3', 'AR')+rho_ratio_*m_sp('FeO', 'AR'))).plot(label='AR', color='k')
(m_sp('Fe2O3', 'FR')/(m_sp('Fe2O3', 'FR')+rho_ratio_*m_sp('FeO', 'FR'))).plot(label='FR', color='k')
ax.text(19,0.991,'AR', fontsize=11)
ax.text(19,0.945,'FR', fontsize=11)
# ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel = 'X, -',
    xlim = (12, 30),
    ylim = (None, 1),
)
fig.tight_layout()
fig.savefig('plots/pilot300W_oxidation.pdf')

In [ ]:
# masses in reactors

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

p.cellSets['controlVolume_AR'].m(phase='solids')['volIntegrate(alpha.solids,rho.solids)'].plot(ax=ax, label='AR', color='k')
p.cellSets['controlVolume_FR'].m(phase='solids')['volIntegrate(alpha.solids,rho.solids)'].plot(ax=ax, label='FR', color='k')
ax.legend()
ax.set(
    xlabel=r'$t$, s',
    ylabel = r'$m$, kg',
    xlim = (12, None)
)
fig.tight_layout()
# fig.savefig('plots/pilot300W_masses.pdf')

## Heat balance
How much energy went to walls, how much energy left through the outlet?

In [ ]:
def getj7(specie):
    if specie in janafs_ilmenite.keys():
        return janafs_ilmenite[specie]
    else:
        return janaf.janaf7Cantera(specie)

def h(specie, T=None):
    if T is None:
        return getj7(specie).dH0f()
    else:
        return getj7(specie).H0(T)

HRRs = {}
for sp in ['CH4', 'CO', 'H2', 'O2']:
    # HRRs[sp] = ilmenite['act'][sp].heat_release_rate(h)
    HRRs[sp] = ilmenite['act'][sp].heat_release_rate(lambda s: h(s, T=900 + 273.15))
    print(f'{sp:3} reaction heat: {HRRs[sp]/1000: 8.3f} kJ/mol')

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))
p.domain.NRR_CO().plot(ax=ax) # mol/s
p.domain.NRR_H2().plot(ax=ax)
p.domain.NRR_O2().plot(ax=ax)

In [ ]:
def get_Tavg_est(location, t, t_window=1):
    """
    Get an estimate of space and time-average of temperature in a reactor at a
    time.

    Location - AR/FR
    t - time
    t_window - time interval to use for average (t-t_window/2, t+t_window/2)
    """
    cellSet_name = 'controlVolume_' + location
    t0 = t-t_window/2
    t1 = t+t_window/2
    Ts = p.cellSets[cellSet_name].Tavgest(phase='solids').loc[t0:t1]
    dt = Ts.index[-1]-Ts.index[0]
    return np.abs(np.trapz(Ts, x=Ts.index, axis=0)/dt)[0]

get_Tavg_est('AR', 1)
# can be used for a more correct calculation of reaction heat, but the effect is very small
# Therefore, reaction heat at 950 °C is assumed for simplicity

### Warning!
Multiplication by 1000 is required since apparently I have kmol/s instead of mol/s

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

# [W]

iFR = (-p.patches['inlet_FR'].hdot(phase='gas'))
iAR = (-p.patches['inlet_AR'].hdot(phase='gas'))
iSL = (-p.patches['inlet_SL'].hdot(phase='gas'))
iDC2 = (-p.patches['inlet_DC2'].hdot(phase='gas'))
iDC1 = (2./3.)*(-p.patches['inlet_DC2'].hdot(phase='gas')).copy() # DC1 = 0.6666*DC2
oAR = p.patches['outlet_AR'].hdot(phase='gas')
oFR = p.patches['outlet_FR'].hdot(phase='gas')

iAll = iFR + iAR + iSL + iDC2 #+ iDC1
oAll = oFR + oAR

heatLeaving = (oAll - iAll).loc[11:]

# dt = heatLeaving.index[-1]-heatLeaving.index[0]
# (np.trapz(heatLeaving, x=heatLeaving.index, axis=0)/dt)[0] / (heatLeaving.index[-1] - heatLeaving.index[0])
dt = oAll.index[-1]-oAll.index[0]
avgoAll = (np.trapz(oAll, x=oAll.index, axis=0)/dt)[0]
print(avgoAll)
# print(iAll)
ax.plot(oAll, label='out')
ax.plot(iAll, label='in')
ax.axhline(avgoAll, color='k', label='avg out')
ax.set(
    xlim=(10,None)
)

ax.legend()

In [ ]:
def get_nonzero_HRR(HRR):
    return HRR[HRR.columns[0]] [ HRR[HRR.columns[0]] != 0. ]

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

def get_total_HRR_averaged(c, t_skip_fraction=0.2):
    """Get total heat release rate."""
    HRR_CO = get_nonzero_HRR(1000*c.domain.NRR_CO()*HRRs['CO'])
    HRR_H2 = get_nonzero_HRR(1000*c.domain.NRR_H2()*HRRs['H2'])
    HRR_O2 = get_nonzero_HRR(1000*c.domain.NRR_O2()*HRRs['O2'])
    # total_HRR = -(HRR_CO[HRR_CO.columns[0]] + HRR_H2[HRR_H2.columns[0]] + HRR_O2[HRR_O2.columns[0]]) # [W] - total heat release rate
    total_HRR = -(HRR_CO + HRR_H2 + HRR_O2) # [W] - total heat release rate
    total_HRR = total_HRR.iloc[int(len(total_HRR)*t_skip_fraction):]
    dt = total_HRR.index[-1]-total_HRR.index[0]
    return np.trapz(total_HRR, x=total_HRR.index, axis=0)/dt

HRR_CO = get_nonzero_HRR(1000*p.domain.NRR_CO()*HRRs['CO'])
HRR_H2 = get_nonzero_HRR(1000*p.domain.NRR_H2()*HRRs['H2'])
HRR_O2 = get_nonzero_HRR(1000*p.domain.NRR_O2()*HRRs['O2'])
# total_HRR = -(HRR_CO[HRR_CO.columns[0]] + HRR_H2[HRR_H2.columns[0]] + HRR_O2[HRR_O2.columns[0]]) # [W] - total heat release rate
total_HRR = -(HRR_CO + HRR_H2 + HRR_O2) # [W] - total heat release rate
total_WHF = - p.patches['walls'].wallHeatFlux()['Q[W]'] # [W] - wall heat flux

ax.plot(-HRR_O2, label=r'$\dot{Q}_{\mathrm{reaction} \, \mathrm{O}_2}$')
ax.plot(-HRR_CO, label=r'$\dot{Q}_{\mathrm{reaction} \, \mathrm{CO}}$')
ax.plot(-HRR_H2, label=r'$\dot{Q}_{\mathrm{reaction} \, \mathrm{H}_2}$')
total_HRR.iloc[1:].plot(ax=ax, label=r'$\dot{Q}_\mathrm{all \, reactions}$')
total_WHF.plot(ax=ax, label=r'$\dot{Q}_\mathrm{wall}$')

ax.legend(ncol = 2, loc = 'center', bbox_to_anchor=(0.5, 0.38))
ax.set(
    xlabel = r'$t$, s',
    ylabel = r'$\dot{Q}$, W',
    ylim = (0,300),
    xlim = (12, 30)
)
fig.tight_layout()
fig.savefig('plots/pilot300W_heat.pdf')

In [ ]:
# Percent of heat, going through the wall

for t in total_WHF.index[1:]:
    index_array = total_HRR.index.to_numpy()
    closest_t = index_array[abs(index_array - t).argmin()]
    print(total_WHF.loc[t] / total_HRR.loc[closest_t])

In [ ]:
t_start = 12
print(f"{total_HRR.loc[t_start:].mean():.1f}")
print(f"{total_WHF.loc[t_start:].mean():.1f}, {total_WHF.loc[t_start:].mean()/total_HRR.loc[t_start:].mean()*100:.2f}")
print(f"{HRR_CO.loc[t_start:].mean():.1f}, {HRR_CO.loc[t_start:].mean()/total_HRR.loc[t_start:].mean()*100:.2f}")
print(f"{HRR_H2.loc[t_start:].mean():.1f}, {HRR_H2.loc[t_start:].mean()/total_HRR.loc[t_start:].mean()*100:.2f}")
print(f"{HRR_O2.loc[t_start:].mean():.1f}, {HRR_O2.loc[t_start:].mean()/total_HRR.loc[t_start:].mean()*100:.2f}")

## Outlet gas concentrations
p.29 [Moldenhauer], Figure 3.4

In [ ]:
Xs = {sp: p.X('FR', sp) for sp in ['CO', 'H2', 'CO2', 'H2O', 'O2']}

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

# ['0072B2', 'E69F00', '009E73', 'CC79A7', 'D55E00', 'k']
arrowprops = dict(facecolor='black', headwidth=6, headlength=4, width=0.5)
annotate_style = dict(va='center', textcoords='offset points', arrowprops=arrowprops)

outl = p.patches['outlet_FR'].mdot_sp()

ax.plot(100*Xs['CO'], color='k')
ax.plot(100*Xs['H2'], color='k')
ax.annotate(r'$\mathrm{CO}$',  (13,1.4), (15,0), **annotate_style)
ax.annotate(r'$\mathrm{H}_2$', (13,0.3), (15,0), **annotate_style)

ax2 = ax.twinx()
ax2.plot(100*Xs['CO2'], color='k')#color = '#009E73')
ax2.plot(100*Xs['H2O'], color='k')#color = '#CC79A7')
ax2.annotate(r'$\mathrm{CO}_2$',  (29,37), (-15,0), ha='right', **annotate_style)
ax2.annotate(r'$\mathrm{H}_2\mathrm{O}$', (29,55), (-15,0), ha='right', **annotate_style)

ax.set(
    xlabel = r'$t$, s',
    ylabel = r'$x_\mathrm{CO}$ and $x_{\mathrm{H2}}$, %',
    ylim = [0,4],
    # xlim = (280, 336)
)

ax2.set(
    xlim = (12,30),
    ylim = [0,62],
    ylabel = r'$x_\mathrm{CO2}$ and $x_{\mathrm{H2O}}$, %',
)
fig.tight_layout()
fig.savefig('plots/pilot300W_outlet_species.pdf')

## Reactor efficiency
p. 29 [Moldenhauer], eqn. (3.16)

In [ ]:
Hi2 = dict(
    H2 = 119953000, # J/kg
    CO =  10103000, # J/kg
)
"""Lower heating values"""

In [ ]:
def efficiency(c):
    """Reactor efficiency over time."""
    outl = c.patches['outlet_FR'].mdot_sp()
    inl = c.patches['inlet_FR'].mdot_sp()

    return 1 - (outl['sum(H2.gas)']*Hi2['H2']+outl['sum(CO.gas)']*Hi2['CO']) \
            / (-inl ['sum(H2.gas)']*Hi2['H2']-inl ['sum(CO.gas)']*Hi2['CO'])

def efficiency_avg(c, t_skip_fraction=0.2):
    """Get a circulation rate for all control surfaces."""
    eff = efficiency(c)
    eff = eff.iloc[int(len(eff)*t_skip_fraction):]
    return np.mean(eff)

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

# 1 - (lower heating value left the system)/(lower heating value entered the system)
# outl = p.patches['outlet_FR'].mdot_sp()
# inl = p.patches['inlet_FR'].mdot_sp()

# gamma_eff = 1 - (outl['sum(H2.gas)']*Hi2['H2']+outl['sum(CO.gas)']*Hi2['CO']) \
#           / (-inl ['sum(H2.gas)']*Hi2['H2']-inl ['sum(CO.gas)']*Hi2['CO'])
# gamma_eff.plot()
efficiency(p).plot(ax=ax, label='efficiency')

ax.set(
    xlabel = '$t$, s',
    ylabel = '$\gamma_\mathrm{eff}$, -',
)

# Mesh sensitivity tests

In [ ]:
ps = [Pilot300W(f'pilot300Wv2_270g_850C_Twall_kN{kN}', offset=0.005) for kN in [1,2,3,4]]
Ncells = [30615, 114465, 486379, 1009681]

In [ ]:
# circulation

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

mdots = [circulation_rate_overall(ps[kN-1], t_skip_fraction=0.7) for kN in [1,2,3,4]]

ax.plot(Ncells, np.abs(1000*np.array(mdots)), '-o')

ax.set(
    xlabel = '$N_\mathrm{cells}$',
    ylabel = '$\dot{m}$, g/s',
)
fig.tight_layout()
fig.savefig('plots/pilot300W_mesh_sensitivity_circulation.pdf')


In [ ]:
# temperature

def T_avg(c, t_start):
    """Get a circulation rate for a given control surface."""
    return c.domain.T()['volAverage(rho.solids,alpha.solids,T.solids)'].loc[t_start:].mean()

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))
for kN in [1,2,3,4]:
    ps[kN-1].domain.register_custom_function_object('T', 'volAverage', )

T_avgs = [T_avg(ps[kN-1], t_start)-273.15 for kN, t_start in zip([1,2,3,4],[150, 40, 20, 12])]

ax.plot(Ncells, T_avgs, '-o')
ax.set(
    xlabel = '$N_\mathrm{cells}$',
    ylabel = '$T_\mathrm{avg}$, °C',
)
fig.tight_layout()
fig.savefig('plots/pilot300W_mesh_sensitivity_temperature.pdf')

In [ ]:
# heat release rate

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

HRR_avgs = [get_total_HRR_averaged(ps[kN-1], t_skip_fraction) for kN, t_skip_fraction in zip([1,2,3,4],[0.5,0.5,0.5,0.5])]

# ax.plot([1,2,3,4], HRR_avgs, '-o')
ax.plot(Ncells, HRR_avgs, '-o')

print('Values are too close and Twall is sampled too infrequently')

In [ ]:
# reactor efficiency

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

eff_avgs = [efficiency_avg(ps[kN-1], t_skip_fraction) for kN, t_skip_fraction in zip([1,2,3,4],[0.5,0.5,0.5,0.5])]

# ax.plot([1,2,3,4], eff_avgs, '-o')
ax.plot(Ncells, np.array(eff_avgs)*100, '-o')
ax.set(
    xlabel = '$N_\mathrm{cells}$',
    ylabel = '$\eta$, %',
)
fig.tight_layout()
fig.savefig('plots/pilot300W_mesh_sensitivity_efficiency.pdf')

## Pressure  
According to Appendix A [Moldenhauer], pressure differences are coming from sensors:  
Difference  
PT12-dc_AR = DC - AR4 = probe10 - probe4  
PT6-dc_FR  = DC - FR4 = probe10 - probe9  

In [ ]:
pressure = p.domain.pressure()

In [ ]:
p_DC_AR = 1e-3*(pressure['10']-pressure['4'])
p_DC_FR = 1e-3*(pressure['10']-pressure['9'])

In [ ]:
fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))

ax.plot(p_DC_AR.index, p_DC_AR.rolling(10000).mean(), label = 'DC-AR')
ax.plot(p_DC_FR.index, p_DC_FR.rolling(10000).mean(), label = 'DC-FR')

ax.set(
    xlabel = r'$t$, s',
    ylabel = '$\Delta p$, kPa'
)

ax.legend()

fig.tight_layout()
fig.savefig('pilot300W_pressure.pdf')

## Leakage and dilution

In [ ]:
# Leakage (FR -> AR)
p.mdot_sp('outlet_AR', 'CO').plot()
p.mdot_sp('outlet_AR', 'H2').plot()
print(-p.mdot_sp('inlet_FR', 'CO').iloc[-1])
print(-p.mdot_sp('inlet_FR', 'H2').iloc[-1])

In [ ]:
# Dilution (AR -> FR)
p.mdot_sp('outlet_FR', 'N2').plot()
p.mdot_sp('outlet_FR', 'O2').plot()
print(-p.mdot_sp('inlet_AR', 'N2').iloc[-1])
print(-p.mdot_sp('inlet_AR', 'O2').iloc[-1])

## Fluidizing gas flow  
The sum does not add up.  
Patch diffusion disabled.  
Continuity error?  
Diffusion flux through control surface?

### SL

In [ ]:
mdot_Ar_SL_AR = p.faceZones['controlSurface_SL_AR'].mdot_sp()['orientedSum(surfaceInterpolate(Ar.gas))']
mdot_Ar_SL_FR = p.faceZones['controlSurface_SL_FR'].mdot_sp()['orientedSum(surfaceInterpolate(Ar.gas))']
mdot_SL = -p.mdot('inlet_SL').iloc[-1]

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))
ax.plot(   1000*mdot_Ar_SL_AR, label='SL_AR')
ax.plot(   1000*mdot_Ar_SL_FR, label='SL_FR')
ax.plot(1000*(mdot_Ar_SL_AR+mdot_Ar_SL_FR), label='total')
ax.axhline(1000*mdot_SL, color='k', linestyle='--', label='inlet_SL')

print(f'SL   : {1000*mdot_SL:6.5f} g/s')

dt = mdot_Ar_SL_AR.index[-1]-mdot_Ar_SL_AR.index[0]
mdot_Ar_SL_AR_avg = np.abs(np.trapz(mdot_Ar_SL_AR, x=mdot_Ar_SL_AR.index, axis=0)/dt)
print(f"SL_AR: {1000*mdot_Ar_SL_AR_avg:6.5f} g/s")

dt = mdot_Ar_SL_FR.index[-1]-mdot_Ar_SL_FR.index[0]
mdot_Ar_SL_FR_avg = np.abs(np.trapz(mdot_Ar_SL_FR, x=mdot_Ar_SL_FR.index, axis=0)/dt)
print(f"SL_FR: {1000*mdot_Ar_SL_FR_avg:6.5f} g/s")

print(f'Percent going to AR: {100*mdot_Ar_SL_AR_avg/(mdot_Ar_SL_AR_avg+mdot_Ar_SL_FR_avg):.3f} %')
print(f'WARNING: Calculation error = {100*(mdot_SL-(mdot_Ar_SL_AR_avg+mdot_Ar_SL_FR_avg))/mdot_SL} %')

ax.set(
    xlabel = r'$t$, s',
    ylabel = r'$\dot{m}$, g/s',
)
ax.legend()


### DC

In [ ]:
mdot_Ar_DC_AR = p.faceZones['controlSurface_DC_AR'].mdot_sp()['orientedSum(surfaceInterpolate(Ar.gas))']#.loc[250:]
mdot_Ar_DC_FR = p.faceZones['controlSurface_DC_FR'].mdot_sp()['orientedSum(surfaceInterpolate(Ar.gas))']#.loc[250:]
mdot_DC2 = -p.mdot('inlet_DC2').iloc[-1]
mdot_DC1 = (2./3.)*mdot_DC2
mdot_DC = mdot_DC1 + mdot_DC2

fig, ax = plt.subplots(figsize=(90*mm, 67.5*mm))
ax.plot(   1000*mdot_Ar_DC_AR, label='DC_AR')
ax.plot(   1000*mdot_Ar_DC_FR, label='DC_FR')
ax.plot(1000*(mdot_Ar_DC_AR+mdot_Ar_DC_FR), label='total')
ax.axhline(1000*mdot_DC, color='k', linestyle='--', label='inlet_DC')

print(f'DC   : {1000*mdot_DC:6.5f} g/s')
dt = mdot_Ar_DC_AR.index[-1]-mdot_Ar_DC_AR.index[0]
mdot_Ar_DC_AR_avg = np.abs(np.trapz(mdot_Ar_DC_AR, x=mdot_Ar_DC_AR.index, axis=0)/dt)
print(f"DC_AR: {1000*mdot_Ar_DC_AR_avg:6.5f} g/s")

dt = mdot_Ar_DC_FR.index[-1]-mdot_Ar_DC_FR.index[0]
mdot_Ar_DC_FR_avg = np.abs(np.trapz(mdot_Ar_DC_FR, x=mdot_Ar_DC_FR.index, axis=0)/dt)
print(f"DC_FR: {1000*mdot_Ar_DC_FR_avg:6.5f} g/s")

print(f'Percent going to AR: {100*mdot_Ar_DC_AR_avg/(mdot_Ar_DC_AR_avg+mdot_Ar_DC_FR_avg):.3f} %')
print(f'WARNING: Calculation error = {100*(mdot_DC-(mdot_Ar_DC_AR_avg+mdot_Ar_DC_FR_avg))/mdot_DC} %')

ax.set(
    xlabel = r'$t$, s',
    ylabel = r'$\dot{m}$, g/s',
)
ax.legend()